In [5]:
import cv2
import threading
from flask import Flask, Response
from ultralytics import YOLO

# 1. Flask 앱 생성
app = Flask(__name__)

# 2. 모델 및 비디오 경로 설정
MODEL_PATH = "/workspace/runs/segment/VAE_Vision/yolo11m_seg_vae_v1/weights/best.pt"  # 학습 완료된 가중치 경로
VIDEO_PATH = "/workspace/Video/Test_1.mp4"                                  # RunPod에 올려둔 원본 비디오 파일

# 모델 로드
model = YOLO(MODEL_PATH)

def generate_frames():
    """
    비디오 스트림을 1프레임씩 읽어 YOLOv11 + ByteTrack 추론 후 
    MJPEG 형식의 바이트 스트림으로 변환하여 yield하는 제너레이터 함수
    """
    while True:
        # stream=True 옵션으로 프레임 단위 연속 추론 진행
        results = model.track(
            source=VIDEO_PATH,
            imgsz=(384, 640),         # 16:9 직사각형 해상도
            tracker="bytetrack.yaml", # ByteTrack 바운딩 박스 보정
            conf=0.40,                # Confidence threshold
            iou=0.6,
            persist=True,             # 객체 ID 추적 지속
            stream=True               # 제너레이터 형태로 프레임 순회
        )
        
        for r in results:
            # 추론 결과(Bounding Box 및 Seg Mask)가 시각화된 프레임 생성
            annotated_frame = r.plot()
            
            # OpenCV 이미지 객체를 JPEG 포맷 바이트로 인코딩
            success, buffer = cv2.imencode('.jpg', annotated_frame)
            if not success:
                continue
                
            frame_bytes = buffer.tobytes()
            
            # MJPEG 스트리밍 프로토콜 규격 전송
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + frame_bytes + b'\r\n')
            
        print("🔄 비디오 재생 완료, 반복 재생합니다.")

@app.route('/video_feed')
def video_feed():
    """웹 브라우저 접속 라우팅 엔드포인트"""
    return Response(
        generate_frames(),
        mimetype='multipart/x-mixed-replace; boundary=frame'
    )

@app.route('/')
def index():
    """기본 접속 확인용 페이지"""
    return "<h1>VAE Vision Real-time Inference Server Running</h1><p>Go to <a href='/video_feed'>/video_feed</a></p>"

# 3. 주피터 노트북에서 커널 락(Lock) 없이 백그라운드 스레드로 Flask 실행
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# 백그라운드 스레드 시작
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("🚀 Flask 추론 스트리밍 서버가 백그라운드에서 실행되었습니다!")
print("👉 브라우저 확인 주소: https://8uiw9mgff5hdr5-5000.proxy.runpod.net/video_feed")

🚀 Flask 추론 스트리밍 서버가 백그라운드에서 실행되었습니다!
👉 브라우저 확인 주소: https://8uiw9mgff5hdr5-5000.proxy.runpod.net/video_feed
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
100.64.1.5 - - [09/Nov/2026 03:52:10] "GET / HTTP/1.1" 200 -



video 1/1 (frame 1/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 10.9ms


100.64.1.5 - - [09/Nov/2026 03:52:18] "GET /video_feed HTTP/1.1" 200 -


video 1/1 (frame 2/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 12.3ms
video 1/1 (frame 3/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.2ms
video 1/1 (frame 4/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.9ms
video 1/1 (frame 5/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.6ms
video 1/1 (frame 6/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.4ms
video 1/1 (frame 7/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.5ms
video 1/1 (frame 8/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.5ms
video 1/1 (frame 9/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.4ms
video 1/1 (frame 10/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.5ms
video 1/1 (frame 11/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 12.8ms
video 1/1 (frame 12/2707) /workspace/Video/Test_1.mp4: 384x640 2 blocks, 3 hooks, 11.9ms
video 1/1 (frame 13/2707) /wo